# Capa Silver - Limpieza y enriquecimiento

Limpieza, casteo de tipos y joins de la tabla de hechos contra las maestras.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date

spark = SparkSession.builder.appName("SilverTransform").getOrCreate()

BASE_VOL = "/Volumes/dbx_diplo_ricardo/default/clase8/clase10"

## Lectura de Bronze

In [ ]:
ventas_df    = spark.read.format("delta").load(f"{BASE_VOL}/bronze/ventas_diarias")
clientes_df  = spark.read.format("delta").load(f"{BASE_VOL}/bronze/clientes")
productos_df = spark.read.format("delta").load(f"{BASE_VOL}/bronze/productos")
tiendas_df   = spark.read.format("delta").load(f"{BASE_VOL}/bronze/tiendas")

print("ventas:",    ventas_df.count())
print("clientes:",  clientes_df.count())
print("productos:", productos_df.count())
print("tiendas:",   tiendas_df.count())

## Limpieza y casts en ventas

In [ ]:
ventas_clean_df = (ventas_df
                   .dropna()
                   .dropDuplicates(["fecha", "tienda_id", "producto_id", "cliente_id"])
                   .withColumn("fecha",           to_date(col("fecha"), "yyyy-MM-dd"))
                   .withColumn("cantidad",        col("cantidad").cast("int"))
                   .withColumn("precio_unitario", col("precio_unitario").cast("double"))
                  )

print("ventas tras limpieza:", ventas_clean_df.count())
ventas_clean_df.printSchema()

## Limpieza de maestras

In [ ]:
clientes_clean  = clientes_df.dropna().dropDuplicates().drop("ciudad")
productos_clean = productos_df.dropna().dropDuplicates()
tiendas_clean   = tiendas_df.dropna().dropDuplicates()

## Enriquecimiento via inner joins

In [ ]:
ventas_enriched_df = (ventas_clean_df
                      .join(clientes_clean,  on="cliente_id",  how="inner")
                      .join(productos_clean, on="producto_id", how="inner")
                      .join(tiendas_clean,   on="tienda_id",   how="inner")
                     )

print("Silver:", ventas_enriched_df.count())
ventas_enriched_df.printSchema()

## Escritura Silver y registro en Unity Catalog

In [ ]:
silver_path = f"{BASE_VOL}/silver/ventas_enriched"

(ventas_enriched_df.write
                   .format("delta")
                   .mode("overwrite")
                   .option("overwriteSchema", "true")
                   .save(silver_path))

spark.sql("CREATE SCHEMA IF NOT EXISTS dbx_diplo_ricardo.silver_clase10")
spark.sql("DROP TABLE IF EXISTS dbx_diplo_ricardo.silver_clase10.silver_ventas")
spark.sql(f"""
    CREATE TABLE dbx_diplo_ricardo.silver_clase10.silver_ventas
    USING DELTA
    LOCATION '{silver_path}'
""")

spark.sql("SHOW TABLES IN dbx_diplo_ricardo.silver_clase10").show(truncate=False)

## Validación

In [ ]:
%sql
SELECT fecha, nombre_tienda, ciudad, nombre_producto, categoria, cantidad, precio_unitario
FROM dbx_diplo_ricardo.silver_clase10.silver_ventas
ORDER BY fecha DESC
LIMIT 10;

In [ ]:
%sql
SELECT categoria, COUNT(*) AS ventas, SUM(cantidad) AS unidades
FROM dbx_diplo_ricardo.silver_clase10.silver_ventas
GROUP BY categoria
ORDER BY unidades DESC;